# Azure ML General Notebook
**Purpose:** Reusable starting point for any Azure ML project.

**Prerequisites before running:**
- `pip install azure-ai-ml azure-identity mltable`
- Fill in your values in the **CONFIG** cell below (Section 1)
- Have a `config.json` in this folder OR fill in credentials manually

---

## Section 1 — Connect to Workspace

**Two options:**
- **Option A (recommended):** Place `config.json` next to this notebook (download it from Azure ML Studio → top-right menu → Download config)
- **Option B (manual):** Fill in your subscription/resource group/workspace below

> ⚠️ The original notebook failed here because `config.json` was not found locally. On an Azure ML Compute Instance it is auto-provided; locally you must supply it yourself.

In [29]:
pip install azure-ai-ml azure-identity mltable mlflow azureml-mlflow

Note: you may need to restart the kernel to use updated packages.


In [30]:
# ── IMPORTS ──────────────────────────────────────────────────────────────────
from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential
from azure.ai.ml import MLClient

# ── AUTHENTICATE ─────────────────────────────────────────────────────────────
# Tries environment/CLI/managed identity first; falls back to browser login
try:
    credential = DefaultAzureCredential()
    credential.get_token("https://management.azure.com/.default")
    print("✓ DefaultAzureCredential succeeded")
except Exception:
    credential = InteractiveBrowserCredential()
    print("✓ Falling back to InteractiveBrowserCredential")

# Esto no esta en el notebook de AML
# On a Compute Instance, config.json is auto-available — no manual IDs needed
# ml_client = MLClient.from_config(credential=credential)
# print(f"✓ Connected to workspace: {ml_client.workspace_name}")

✓ DefaultAzureCredential succeeded


In [31]:
# ── OPTION A: config.json (recommended) ──────────────────────────────────────
# Download config.json from Azure ML Studio and place it next to this notebook.
# Then run this cell.

ml_client = MLClient.from_config(credential=credential)

# ── OPTION B: Manual credentials (fallback) ───────────────────────────────────
# Fill in your values if you don't have config.json

# SUBSCRIPTION_ID = "YOUR-SUBSCRIPTION-ID"   # e.g. "xxxxxxxx-xxxx-xxxx-xxxx-xxxxxxxxxxxx"
# RESOURCE_GROUP  = "YOUR-RESOURCE-GROUP"     # e.g. "rg-mlworkspace"
# WORKSPACE_NAME  = "YOUR-WORKSPACE-NAME"     # e.g. "my-aml-workspace"

# ml_client = MLClient(
#     credential=credential,
#     subscription_id=SUBSCRIPTION_ID,
#     resource_group_name=RESOURCE_GROUP,
#     workspace_name=WORKSPACE_NAME,
# )

# print(f"✓ Connected to workspace: {ml_client.workspace_name}")

Found the config file in: /config.json
Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


---
## Section 2 — Datastores

In [ ]:
# List all datastores in the workspace
print("Datastores in workspace:")
for ds in ml_client.datastores.list():
    print(" -", ds.name)



In [ ]:
# Create a new Blob datastore (only run this when you want to register a new storage)
# Fill in YOUR-STORAGE-ACCOUNT-NAME and YOUR-ACCOUNT-KEY before running

from azure.ai.ml.entities import AzureBlobDatastore, AccountKeyConfiguration

store = AzureBlobDatastore(
    name="blob_training_data",
    description="Blob Storage for training data",
    account_name="YOUR-STORAGE-ACCOUNT-NAME",
    container_name="training-data",
    credentials=AccountKeyConfiguration(
        account_key="YOUR-ACCOUNT-KEY"
    ),
)

ml_client.datastores.create_or_update(store)
print(f"✓ Datastore '{store.name}' registered")

---
## Section 3 — Data Assets

In [ ]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

# URI_FILE — points to a single CSV file (uploaded automatically to default datastore)
my_data = Data(
    path="/home/azureuser/cloudfiles/code/Users/Castle2696/azure-ml-labs/data/diabetes.csv",
    type=AssetTypes.URI_FILE,
    description="Single CSV file data asset",
    name="diabetes-local"
)
ml_client.data.create_or_update(my_data)
print("✓ URI_FILE data asset registered")

In [ ]:
# URI_FOLDER — points to a folder in a registered datastore
my_data = Data(
    path="azureml://datastores/blob_training_data/paths/data-asset-path/",
    type=AssetTypes.URI_FOLDER,
    description="Folder in blob datastore",
    name="diabetes-datastore-path"
)
ml_client.data.create_or_update(my_data)
print("✓ URI_FOLDER data asset registered")

In [ ]:
# MLTABLE — points to a folder containing an MLTable file
# The folder must contain a file named exactly 'MLTable' (no extension)
my_data = Data(
    path="/home/azureuser/cloudfiles/code/Users/Castle2696/azure-ml-labs/data/",
    type=AssetTypes.MLTABLE,
    description="MLTable pointing to diabetes data folder",
    name="diabetes-table"
)
ml_client.data.create_or_update(my_data)
print("✓ MLTABLE data asset registered")

In [ ]:
# List all registered data assets
print("Registered data assets:")
for ds in ml_client.data.list():
    print(" -", ds.name)

In [ ]:
# Read an MLTable data asset into a Pandas DataFrame
# Requires: pip install mltable
import mltable

registered_data_asset = ml_client.data.get(name="diabetes-table", version=1)
tbl = mltable.load(f"azureml:/{registered_data_asset.id}")
df = tbl.to_pandas_dataframe()
df.head(5)

In [ ]:
import mltable

latest = max(
    ml_client.data.list(name="diabetes-table"),
    key=lambda x: int(x.version)
)
print(f"Using version: {latest.version}")
tbl = mltable.load(f"azureml:/{latest.id}")
df = tbl.to_pandas_dataframe()
df.head(5)

---
## Section 4 — Compute

Esto tambien se puede hacer desde Cl

In [ ]:
from azure.ai.ml.entities import AmlCompute

cpu_compute_target = "aml-cluster"

try:
    cpu_cluster = ml_client.compute.get(cpu_compute_target)
    print(f"✓ Compute '{cpu_compute_target}' already exists — reusing it")
except Exception:
    print(f"Creating compute cluster '{cpu_compute_target}'...")
    cpu_cluster = AmlCompute(
        name=cpu_compute_target,
        type="amlcompute",
        size="STANDARD_DS11_V2",
        min_instances=0,
        max_instances=2,
        idle_time_before_scale_down=120,
        tier="Dedicated",
    )
    cpu_cluster = ml_client.compute.begin_create_or_update(cpu_cluster)
    print(f"✓ Compute '{cpu_compute_target}' created")

---
## Section 5.1 — Run a Training Job + Environment custom and curated

Important! dataset + script + conda-env.yml TOGETHER

In [ ]:
import os
os.makedirs("src", exist_ok=True)

In [ ]:
%%writefile src/diabetes-training.py
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

print("Loading Data...")
diabetes = pd.read_csv('diabetes.csv')

X = diabetes[['Pregnancies','PlasmaGlucose','DiastolicBloodPressure',
              'TricepsThickness','SerumInsulin','BMI','DiabetesPedigree','Age']].values
y = diabetes['Diabetic'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=0)

reg = 0.01
print(f'Training LogisticRegression with regularization rate = {reg}')
model = LogisticRegression(C=1/reg, solver="liblinear").fit(X_train, y_train)

y_hat = model.predict(X_test)
acc = np.average(y_hat == y_test)
print('Accuracy:', acc)

y_scores = model.predict_proba(X_test)
auc = roc_auc_score(y_test, y_scores[:, 1])
print('AUC:', auc)

---
## Section 5.2 — Environments

In [ ]:
# List all environments (curated + custom)
print("Available environments:")
for env in ml_client.environments.list():
    print(" -", env.name)

Creating the custom environment. 

In [ ]:
from azure.ai.ml.entities import Environment

env_docker_image = Environment(
    image="mcr.microsoft.com/azureml/openmpi3.1.2-ubuntu18.04",
    name="docker-image-example",
    description="Environment created from a Docker image.",
)
ml_client.environments.create_or_update(env_docker_image)

First try to train the job with a custom environment

In [ ]:
from azure.ai.ml import command

# configure job
job = command(
    code="./src",
    command="python diabetes-training.py",
    environment="docker-image-example:1",
    compute="aml-cluster",
    display_name="diabetes-train-custom-env",
    experiment_name="diabetes-training"
)

# submit job
returned_job = ml_client.create_or_update(job)
aml_url = returned_job.studio_url
print("Monitor your job at", aml_url)

Custom an Environment

In [ ]:
%%writefile src/conda-env.yml
name: basic-env-cpu
channels:
  - conda-forge
dependencies:
  - python=3.11
  - scikit-learn
  - pandas
  - numpy
  - matplotlib

In [ ]:
from azure.ai.ml.entities import Environment

# Custom environment: base Docker image + conda dependencies
env_docker_conda = Environment(
    image="mcr.microsoft.com/azureml/openmpi3.1.2-ubuntu18.04",
    conda_file="./src/conda-env.yml",
    name="docker-image-plus-conda-example",
    description="Base Docker image with conda dependencies",
)
ml_client.environments.create_or_update(env_docker_conda)
print(f"✓ Environment '{env_docker_conda.name}' registered")

Running the job with new custom environment

In [ ]:
from azure.ai.ml import command

job = command(
    code="./src",
    command="python diabetes-training.py",
    environment="AzureML-sklearn-1.5@latest",  # Use curated env (safe default)
    compute="aml-cluster",
    display_name="diabetes-pythonv2-train",
    experiment_name="diabetes-training"
)

returned_job = ml_client.jobs.create_or_update(job)   # FIX: use ml_client.jobs, not ml_client directly
print("Monitor your job at:", returned_job.studio_url)

---
## Section 6 — MLflow Tracking

Prepare the Data

In [ ]:
import pandas as pd

print("Reading data...")
df = pd.read_csv('/home/azureuser/cloudfiles/code/Users/Castle2696/azure-ml-labs/data/diabetes.csv')
df.head()

In [ ]:
print("Splitting data...")
X, y = df[['Pregnancies','PlasmaGlucose','DiastolicBloodPressure','TricepsThickness','SerumInsulin','BMI','DiabetesPedigree','Age']].values, df['Diabetic'].values

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=0)

Create an MLflow

In [ ]:
import mlflow
import azureml.mlflow 

# The azureml-mlflow package registers the azureml:// URI handler
mlflow_tracking_uri = ml_client.workspaces.get(ml_client.workspace_name).mlflow_tracking_uri
mlflow.set_tracking_uri(mlflow_tracking_uri)
mlflow.set_experiment("mlflow-experiment-diabetes")

print("Tracking URI:", mlflow.get_tracking_uri())
print("Experiment:", mlflow.get_experiment_by_name("mlflow-experiment-diabetes"))

Train and track models

In [ ]:
from sklearn.linear_model import LogisticRegression

with mlflow.start_run():
    mlflow.sklearn.autolog()

    model = LogisticRegression(C=1/0.1, solver="liblinear").fit(X_train, y_train)

Logistic Regression

Agregando mlflow.logsss. Muy Importante!

In [ ]:
from sklearn.linear_model import LogisticRegression
import numpy as np

with mlflow.start_run():
    model = LogisticRegression(C=1/0.1, solver="liblinear").fit(X_train, y_train)

    y_hat = model.predict(X_test)
    acc = np.average(y_hat == y_test)

    mlflow.log_param("regularization_rate", 0.1) #Configurables
    mlflow.log_metric("Accuracy", acc) #Configurables

The reason why you'd want to track models, could be to compare the results of models you train with different hyperparameter values. For example, you just trained a logistic regression model with a regularization rate of 0.1. Now, train another model, but this time with a regularization rate of 0.01. Since you're also tracking the accuracy, you can compare and decide which rate results in a better performing model.

In [ ]:
from sklearn.linear_model import LogisticRegression
import numpy as np

with mlflow.start_run():
    model = LogisticRegression(C=1/0.01, solver="liblinear").fit(X_train, y_train)

    y_hat = model.predict(X_test)
    acc = np.average(y_hat == y_test)

    mlflow.log_param("regularization_rate", 0.01)
    mlflow.log_metric("Accuracy", acc)

Decision tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier
import numpy as np

with mlflow.start_run():
    model = DecisionTreeClassifier().fit(X_train, y_train)

    y_hat = model.predict(X_test)
    acc = np.average(y_hat == y_test)

    mlflow.log_param("estimator", "DecisionTreeClassifier")
    mlflow.log_metric("Accuracy", acc)

Logs Artifacts
Finally, let's try to log an artifact. An artifact can be any file. For example, you can plot the ROC curve and store the plot as an image. The image can be logged as an artifact.

Run the following cell to log a parameter, metric, and an artifact.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_curve
import matplotlib.pyplot as plt
import numpy as np

with mlflow.start_run():
    # autolog handles params, metrics, and model automatically
    mlflow.sklearn.autolog()

    model = DecisionTreeClassifier().fit(X_train, y_train)
    y_hat = model.predict(X_test)
    acc = np.average(y_hat == y_test)

    # Log ROC curve as figure — avoids log_artifact file path issue
    y_scores = model.predict_proba(X_test)
    fpr, tpr, _ = roc_curve(y_test, y_scores[:, 1])
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot([0, 1], [0, 1], 'k--')
    ax.plot(fpr, tpr)
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title('ROC Curve')

    # log_figure instead of log_artifact — no file path, no version conflict
    mlflow.log_figure(fig, "ROC-Curve.png")
    mlflow.log_metric("Accuracy", acc)

    print(f"✓ Run logged — Accuracy: {acc:.4f}")

---
## Section 8 — AutoML

Prepare data

In [45]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes
import mltable

my_data = Data(
    path="/home/azureuser/cloudfiles/code/Users/Castle2696/azure-ml-labs/data",
    type=AssetTypes.MLTABLE,
    description="MLTable pointing to diabetes data folder",
    name="diabetes-table"
)
ml_client.data.create_or_update(my_data)
print("✓ MLTABLE data asset registered")

latest = max(
    ml_client.data.list(name="diabetes-table"),
    key=lambda x: int(x.version)
)
data_asset = ml_client.data.get(name="diabetes-table", version=latest.version)
print(f"Asset ID: {data_asset.id}")
print(f"Using version: {latest.version}")
#tbl = mltable.load(f"azureml:/{latest.id}")
tbl = mltable.load(data_asset.path)
df = tbl.to_pandas_dataframe()
df.head(5)

Uploading data (0.52 MBs): 100%|██████████| 518703/518703 [00:00<00:00, 8244836.81it/s]




✓ MLTABLE data asset registered
Asset ID: /subscriptions/735ef05d-6290-4c6c-bbb5-2e8ba6ecb52e/resourceGroups/rg-dp100-l3107efa2fa8940acbf/providers/Microsoft.MachineLearningServices/workspaces/mlw-dp100-l3107efa2fa8940acbf/data/diabetes-table/versions/7
Using version: 7


,PatientID,Pregnancies,PlasmaGlucose,DiastolicBloodPressure,TricepsThickness,SerumInsulin,BMI,DiabetesPedigree,Age,Diabetic
0,1354778,0,171,80,34,23,43.509726,1.213191,21,False
1,1147438,8,92,93,47,36,21.240576,0.158365,23,False
2,1640031,7,115,47,52,35,41.511523,0.079019,23,False
3,1883350,9,103,78,25,304,29.582192,1.282870,43,True
4,1424119,1,85,59,27,35,42.604536,0.549542,22,False


Configure automated machine learning job

Now, you're ready to configure the automated machine learning experiment.

When you run the code below, it will create an automated machine learning job that:

- Uses the compute cluster named *aml-cluster*
- Sets *Diabetic* as the target column
- Sets *accuracy* as the primary metric
- Times out after *60* minutes of total training time
- Trains a maximum of *5* models
- No model will be trained with the *LogisticRegression* algorithm

In [47]:
from azure.ai.ml import automl, Input
from azure.ai.ml.constants import AssetTypes

# Input must be an MLTable for AutoML
my_training_data_input = Input(
    type=AssetTypes.MLTABLE,
    path="azureml:diabetes-table:7"
)

classification_job = automl.classification(
    compute="aml-cluster",
    experiment_name="auto-ml-class-dev",
    training_data=my_training_data_input,
    target_column_name="Diabetic",
    primary_metric="accuracy",
    n_cross_validations=5,
    enable_model_explainability=True
)

classification_job.set_limits(
    timeout_minutes=60,
    trial_timeout_minutes=20,
    max_trials=5,
    enable_early_termination=True,
)

classification_job.set_training(
    blocked_training_algorithms=["LogisticRegression"],
    enable_onnx_compatible_models=True
)

# Submit
returned_job = ml_client.jobs.create_or_update(classification_job)
print("Monitor AutoML job at:", returned_job.studio_url)

Monitor AutoML job at: https://ml.azure.com/runs/zen_lion_xh1v969n4f?wsid=/subscriptions/735ef05d-6290-4c6c-bbb5-2e8ba6ecb52e/resourcegroups/rg-dp100-l3107efa2fa8940acbf/workspaces/mlw-dp100-l3107efa2fa8940acbf&tid=e43b0362-e932-4a58-9c0d-3071db3f47a8
